# Benchmarking Fleets compute profiles with SQD IEF-PCM

**Bigger compute profile → faster SQD post-processing.** We run the same methanol
`(14e, 12o)` workload as `deploy_and_run.ipynb` on three Fleets profiles and compare.

Only the **post-processing** stage parallelizes: for each S-CORE iteration it runs
`number_of_batches` independent eigenstate solves, one per CPU worker. That is the stage
a bigger profile speeds up (it dominated at ~282 s in `deploy_and_run.ipynb`), so to make
the profile the *only* variable we:

- **bypass the QPU** — every run reuses one cached set of LUCJ counts, so all profiles
  diagonalize identical input;
- **fix `number_of_batches = 48`** — enough parallel work to keep the big profiles busy.

| Profile | CPU | Batches ÷ CPU | Expected |
| --- | --- | --- | --- |
| `8x32` | 8 | 6 waves | baseline |
| `16x128` | 16 | 3 waves | ~2× faster |
| `48x768` | 48 | 1 wave | ~6× faster |


## 1. Connect


In [ ]:
from qiskit_ibm_catalog import QiskitServerless, QiskitFunction

serverless = QiskitServerless(token="MY_TOKEN", instance="MY_INSTANCE")

## 2. Workload

Identical to `deploy_and_run.ipynb`, except `number_of_batches = 48`.


In [ ]:
molecule = {
    "atom": """
    O -0.04559 -0.75076 -0.00000;
    C -0.04844 0.65398 -0.00000;
    H 0.85330 -1.05128 -0.00000;
    H -1.08779 0.98076 -0.00000;
    H 0.44171 1.06337 0.88811;
    H 0.44171 1.06337 -0.88811
    """,
    "basis": "cc-pvdz",
    "spin": 0,
    "charge": 0,
    "verbosity": 0,
    "number_of_active_orb": 12,
    "number_of_active_alpha_elec": 7,
    "number_of_active_beta_elec": 7,
    "avas_selection": ["%d O %s" % (0, x) for x in ["2s", "2px", "2py", "2pz"]]
    + ["%d C %s" % (1, x) for x in ["2s", "2px", "2py", "2pz"]]
    + ["%d H 1s" % k for k in [2, 3, 4, 5]],
}

solvent_options = {"method": "IEF-PCM", "eps": 78.3553}

initial_layout = [0, 14, 18, 19, 20, 33, 39, 40, 41, 53, 60, 61]
initial_layout += [2, 3, 4, 15, 22, 23, 24, 34, 43, 44, 45, 54]
lucj_options = {
    "initial_layout": initial_layout,
    "dynamical_decoupling_choice": True,
    "twirling_choice": True,
    "number_of_shots": 200000,
    "optimization_level": 2,
}

sqd_options = {
    "sqd_iterations": 3,
    "number_of_batches": 48,  # the parallel width of the sweep
    "samples_per_batch": 200,  # small -> each batch is a fast, sparse solve
    "max_davidson_cycles": 200,
    "blas_threads_per_worker": None,  # None = auto
}

## 3. Upload with a sizes catalog

The `sizes_map` declares the three profiles we sweep. The first run samples the LUCJ
counts on the QPU and caches them (function-scoped); every later run reuses that file and
skips the QPU.


In [ ]:
SIZES_MAP = {"s": "8x32", "m": "16x128", "l": "48x768"}  # CPU x memory(GB)

benchmark = QiskitFunction(
    title="sqd_pcm_benchmark_fleets",
    entrypoint="sqd_pcm_entrypoint.py",
    working_dir="./source_files/",
    runner="fleets",
    sizes_map=SIZES_MAP,
    default_size="s",
)
serverless.upload(benchmark)
benchmark = serverless.load("sqd_pcm_benchmark_fleets")
print("sizes:", benchmark.sizes())

## 4. Run the sweep

One job per profile, all diagonalizing the same cached counts.


In [ ]:
import time


def submit(size):
    return benchmark.run(
        function_size=size,
        backend_name="ibm_kingston",  # unused on the cached-counts path
        molecule=molecule,
        solvent_options=solvent_options,
        lucj_options=lucj_options,
        sqd_options=sqd_options,
        count_dict_file_name="methanol_counts.txt",
    )


jobs = {label: submit(label) for label in SIZES_MAP}
for label, job in jobs.items():
    print(f"submitted {label} ({SIZES_MAP[label]}): {job.job_id}")

Collect the post-processing time from each result once the jobs finish.


In [ ]:
def collect(label, job):
    while job.status() not in ("DONE", "ERROR"):
        time.sleep(5)
    meta = job.result()["metadata"]
    bd = meta.get("post_processing_breakdown", {})
    cpu = int(SIZES_MAP[label].split("x")[0])
    return {
        "profile": SIZES_MAP[label],
        "cpu": cpu,
        "post_processing_s": meta["resources_usage"]["RUNNING: POST_PROCESSING"]["CPU_TIME"],
        "solve_s": bd.get("parallel_solve_time", float("nan")),
        "prep_s": bd.get("serial_prep_time", float("nan")),
    }


records = sorted((collect(l, j) for l, j in jobs.items()), key=lambda r: r["cpu"])
for r in records:
    print(f"{r['profile']:>8} ({r['cpu']:>2} CPU): {r['post_processing_s']:7.1f} s")

## 5. Result

Each bar splits post-processing into the **parallel solve** (shrinks with the profile)
and the **serial prep** — config recovery between S-CORE iterations, the flat *Amdahl
floor* no profile removes.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = [f"{r['profile']}\n({r['cpu']} CPU)" for r in records]
prep = np.array([r["prep_s"] for r in records])
solve = np.array([r["solve_s"] for r in records])
totals = prep + solve
speedup = totals[0] / totals

x = np.arange(len(records))
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(x, prep, color="#c2b0a3", label="serial prep (Amdahl floor)")
ax.bar(x, solve, bottom=prep, color="#2a5563", label="parallel solve")
for xi, total, sp in zip(x, totals, speedup):
    ax.text(xi, total, f"{total:.0f} s\n{sp:.1f}x", ha="center", va="bottom", fontsize=10)

ax.set_xticks(x, labels)
ax.set_ylabel("SQD post-processing time (s)")
ax.set_title("Fleets compute profile vs. SQD post-processing\nmethanol (14e, 12o), 48 batches")
ax.legend(loc="upper right", fontsize=9)
ax.margins(y=0.18)
fig.tight_layout()
plt.show()

## Takeaway

The parallel solve falls with more cores; the serial prep floor does not. Speedup is
**sub-linear** and bounded by that floor — and once you have more cores than batches,
extra cores stop adding concurrency. Match the profile to `number_of_batches`.
